In [11]:
# ETL (인덱싱, 청킹, 임베딩, 저장)
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore


In [16]:
raw_text = """[스파르타대학교 수강신청 및 성적 장학 규정]
제1조(신청 학점): 학생은 학기당 최소 12학점에서 최대 19학점까지 수강 신청할 수 있습니다.
제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.
제3조(성적 장학금): 직전 학기 15학점 이상을 이수하고 평점 평균 3.5 이상을 취득한 학생 중 선발합니다.
단, F 학점이 포함된 과목이 있는 경우 성적 장학금 지급 대상에서 제외됩니다."""

# 청킹 
text_splitter = RecursiveCharacterTextSplitter(
    #separator="",
    chunk_size=90,
    chunk_overlap=20
)

split_docs = text_splitter.create_documents([raw_text])

len(split_docs)

4

In [17]:
for i, doc in enumerate(split_docs, 1):
    print(f"[청크: {i}] {doc.page_content}")

[청크: 1] [스파르타대학교 수강신청 및 성적 장학 규정]
제1조(신청 학점): 학생은 학기당 최소 12학점에서 최대 19학점까지 수강 신청할 수 있습니다.
[청크: 2] 제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.
[청크: 3] 제3조(성적 장학금): 직전 학기 15학점 이상을 이수하고 평점 평균 3.5 이상을 취득한 학생 중 선발합니다.
[청크: 4] 단, F 학점이 포함된 과목이 있는 경우 성적 장학금 지급 대상에서 제외됩니다.


In [18]:
# 임베딩 - 문장 -> 실수 벡터 -> 적재 
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY2"))

vector_store = InMemoryVectorStore.from_documents(
    documents=split_docs,
    embedding=embedding_model
)

In [21]:
# 검색(Retrieval)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

user_query = "직전 학기 학점이 4.1이에요. 다음 학기에 최대 몇 학점까지 신청할 수 있나요?"

retrieved_results = retriever.invoke(user_query)

for i, doc in enumerate(retrieved_results, 1):
    print(f"[{i}번] {doc.page_content}")

[1번] 제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.


In [22]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
    api_key=os.getenv("OPENAI_API_KEY2")
) 

prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 [참고문서]의 내용마을 근거하여 답변하세요. [참고문서]에서 없는 내용은 모른다라고 답변하세요.\n\n[참고문서]:\n{context}"),
    ("user", "{question}")
])

parser = StrOutputParser()

chain = prompt | model | parser
response = chain.invoke({
    "context": retrieved_results[0].page_content,
    "question": user_query
})

print(response)

다음 학기에 최대 21학점까지 신청할 수 있습니다.
